**Contact Boundary and Gap Function in NGSolve**

In [84]:
import numpy as np
import matplotlib.pyplot as plt

from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from ngsolve.solvers import NewtonMinimization
from netgen.webgui import Draw as DrawGeo

**Parallel Contact Boundary**

In [85]:
f1 = MoveTo(0, 0).Rectangle(1, 1).Face()
f1.edges.Max(Y).name = "contact_1"
f1.name = "body_1"

f2 = MoveTo(0.0, 1).Rectangle(1, 1).Face()
f2 = f2.Rotate(Axis((0.5, 1.5, 0), (0, 0, 1)), 90)
f2.edges.Min(Y).name = "contact_2"
f2.name = "body_2"

geo = Compound([f1, f2])
geo = OCCGeometry(geo, dim = 2)

mesh = Mesh(geo.GenerateMesh(maxh=0.1, quad_dominated=True))

Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
fes = VectorH1(mesh, order=2, dirichlet="left|right|bottom|top")
u = fes.TrialFunction()
v = fes.TestFunction()

# Define a displacement field
offset = -0.1
gfu = GridFunction(fes)
gfu.Set((0, offset), definedon=mesh.Materials("body_2"))
Draw(gfu, mesh, "displacement", deformation = gfu)

# Define Contact
master = mesh.Boundaries("contact_1")
slave = mesh.Boundaries("contact_2")
contact = ContactBoundary(master, slave)

# Update contact with current displacement
contact.Update(gfu, None, 10, 1)

# Get gap vector and normal
gap_vec_master = contact.gap
n_master = contact.normal

gap_n_master = InnerProduct(gap_vec_master, n_master)

# Calculate the gap distance
gap_integral = Integrate(gap_n_master, master)
print(f"Gap distance: {gap_integral:.6f}")


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Gap distance: -0.100000


For parallel contact boundaries this approach yields the correct result.

-----------------------------------------------------------

**Curved Contact Boundary**

In [88]:
square = MoveTo(-0.5, 0).Rectangle(1, 1).Face()
square.edges.Max(Y).name = "contact_square"
square.name = "square"

circle = Circle((0, 1.5), 0.5).Face()
circle.edges.name = "contact_circle"
circle.name = "circle"

geo = Compound([square, circle])
geo = OCCGeometry(geo, dim = 2)

mesh = Mesh(geo.GenerateMesh(maxh=0.1, quad_dominated=False))

Draw(mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [89]:
fes = VectorH1(mesh, order=3)
u = fes.TrialFunction()
v = fes.TestFunction()

offset = -0.1

gfu = GridFunction(fes)
gfu.Set((0, offset), definedon=mesh.Materials("circle"))

Draw(gfu, mesh, "displacement", deformation = gfu)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [90]:
# Define Contact
master = mesh.Boundaries("contact_square")
slave = mesh.Boundaries("contact_circle")
contact = ContactBoundary(master, slave)

# Update contact with current displacement
contact.Update(gfu, None, 10, 1)

In [92]:
# Get gap vector and normal
gap_vec_master = contact.gap
n_master = contact.normal

gap_n_master = InnerProduct(gap_vec_master, n_master)

# Calculate the gap distance
gap_integral = Integrate(gap_n_master, master)
print(f"Gap distance: {gap_integral}")

Gap distance: -0.017480268959969618


Inner product of gao_vec_mastera and gap normal vector is integrated over the total master contact boundary.
Since the circle penetrate the square only on a small part of the master boundary, the gap integral does not correspond to the actual maximum penetration depth.